# aqui rodamos 14 meses com o prompt selecionado dos testes o prompt 13 mais RECALL 3.2 avg.


prompt 13:

changed max keep to 4

The Strategy is distinct:

Prompt 12 (Strict): Avg ~2.5 (High Precision)

Prompt 13 (Safety): Avg ~3.2 (High Recall)

# Helpers

In [1]:
import json
import re
import random
import statistics
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
_RUN_RE = re.compile(
    r"llama_drop_only_(?P<month>adzuna_month\d{2})_(?P<start>\d+)_(?P<stop>\d+)_job(?P<jobid>\d+)_task(?P<taskid>\d+)_\d{8}_\d{6}\.jsonl$"
)

IT_PATTERNS = [
    r"\bsoftware\b", r"\bdeveloper\b", r"\bengineer\b", r"\bdata\b", r"\bml\b", r"\bai\b",
    r"\bcloud\b", r"\bcyber\b", r"\bsecurity\b", r"\bnetwork\b", r"\bsystems?\b",
    r"\bdatabase\b", r"\bit\b", r"\bdevops\b"
]
_IT_RE = re.compile("|".join(IT_PATTERNS), flags=re.I)

def _is_it_role(title: str) -> bool:
    if not title:
        return False
    return bool(_IT_RE.search(title))


def _infer_npz_from_jsonl(jsonl_path: Path) -> Path:
    """
    NEW layout:
      .../llm_negative_selection/<EMBED>/<adzuna_monthXX>/llama_drop_only_....jsonl
    NPZ sits at:
      .../llm_negative_selection/<EMBED>/<adzuna_monthXX>.npz
    """
    m = _RUN_RE.search(jsonl_path.name)
    if not m:
        raise ValueError(f"JSONL filename doesn't match expected pattern: {jsonl_path.name}")

    month = m.group("month")
    embed_root = jsonl_path.parent.parent  # .../llm_negative_selection/<EMBED>
    return embed_root / f"{month}.npz"


def _get_job_ids(z) -> np.ndarray:
    """
    Supports both schemas:
      new: job_ids
      old: job_id
    Returns np.ndarray[str]
    """
    if "job_ids" in z.files:
        return z["job_ids"].astype(str)
    if "job_id" in z.files:
        return z["job_id"].astype(str)
    raise KeyError(f"NPZ missing job id key. Need job_ids or job_id. Have={sorted(z.files)}")


def _load_npz_lookup(npz_path: Path):
    """
    Expects stage3-prep NPZ keys (canonical):
      job_ids (or job_id), job_ad_title, job_desc, job_tasks, domain, job_sector_category, job_description
    """
    with np.load(npz_path, allow_pickle=True) as z:
        job_ids = _get_job_ids(z)

        # canonical (your stage3-prep)
        job_ad_title = z["job_ad_title"]
        job_desc = z["job_desc"]
        job_tasks = z["job_tasks"]

        domain = z["domain"] if "domain" in z.files else None
        job_sector_category = z["job_sector_category"] if "job_sector_category" in z.files else None
        job_description = z["job_description"] if "job_description" in z.files else None

    lookup = {}
    for i, jid in enumerate(job_ids):
        lookup[jid] = {
            "job_ad_title": None if job_ad_title[i] is None else str(job_ad_title[i]),
            "job_desc": None if job_desc[i] is None else str(job_desc[i]),
            "job_tasks": None if job_tasks[i] is None else str(job_tasks[i]),
            "job_description": None if job_description is None or job_description[i] is None else str(job_description[i]),
            "domain": None if domain is None or domain[i] is None else str(domain[i]),
            "job_sector_category": None if job_sector_category is None or job_sector_category[i] is None else str(job_sector_category[i]),
        }
    return lookup


# ---------------------------------------------------------------------
# Main: report generator
# ---------------------------------------------------------------------
def gen_report(jsonl_path: str, *, npz_path: str | None = None, sample_n: int = 30, seed: int = 0):
    """
    jsonl_path: path to llama_drop_only_*.jsonl
    npz_path  : optional explicit context NPZ. If None, inferred from jsonl name/dir.
    """
    random.seed(seed)

    jsonl_path = Path(jsonl_path)
    if not jsonl_path.exists():
        raise FileNotFoundError(f"Missing JSONL: {jsonl_path}")

    inferred_npz = _infer_npz_from_jsonl(jsonl_path)
    npz_path = Path(npz_path) if npz_path else inferred_npz
    if not npz_path.exists():
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    # report folder: put it under .../llm_negative_selection/<EMBED>/evaluation_reports/
    embed_root = jsonl_path.parent.parent  # .../llm_negative_selection/<EMBED>
    report_dir = embed_root / "evaluation_reports"
    report_dir.mkdir(parents=True, exist_ok=True)

    print("JSONL:", jsonl_path)
    print("NPZ :", npz_path)
    print("OUT :", report_dir)

    lookup = _load_npz_lookup(npz_path)

    before_counts = []
    after_counts = []
    kept_titles = Counter()
    domain_kept = defaultdict(list)

    it_leak = 0
    total_kept = 0
    empty_outputs = 0

    samples = []
    seen_sample_ids = set()

    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                r = json.loads(line)
            except Exception:
                continue

            jid = str(r.get("job_id", "")).strip()
            if not jid:
                continue

            ctx = lookup.get(jid, {})

            cand = r.get("candidates") or []
            final = r.get("final") or []

            before_counts.append(len(cand))
            after_counts.append(len(final))

            if len(final) == 0:
                empty_outputs += 1

            for t in final:
                kept_titles[t] += 1
                total_kept += 1
                if _is_it_role(t):
                    it_leak += 1

            dom = ctx.get("domain") or "UNKNOWN"
            domain_kept[dom].append(len(final))

            # sample gate
            if len(samples) < sample_n and jid not in seen_sample_ids and random.random() < 0.05:
                seen_sample_ids.add(jid)
                samples.append({
                    "job_id": jid,
                    "job_ad_title": ctx.get("job_ad_title"),
                    "domain": ctx.get("domain"),
                    "sector": ctx.get("job_sector_category"),
                    "job_desc": ctx.get("job_desc"),
                    "kept": final,
                    "dropped": r.get("drop") or [],
                })

    if not before_counts:
        raise RuntimeError("No valid rows parsed from JSONL.")

    before_avg = statistics.mean(before_counts)
    after_avg = statistics.mean(after_counts)
    drop_rate = 1.0 - (after_avg / before_avg if before_avg else 0.0)
    it_share = it_leak / max(total_kept, 1)

    metrics = {
        "jobs": len(before_counts),
        "avg_candidates_before": round(before_avg, 3),
        "avg_candidates_after": round(after_avg, 3),
        "drop_rate": round(drop_rate, 4),
        "empty_outputs_percent": round(100.0 * empty_outputs / len(before_counts), 3),
        "it_leakage_share": round(it_share, 4),
        "min_kept": int(min(after_counts)),
        "max_kept": int(max(after_counts)),
    }

    domain_summary = {d: round(statistics.mean(v), 3) for d, v in domain_kept.items()}

    report = {
        "run": {
            "jsonl": str(jsonl_path),
            "npz": str(npz_path),
            "generated_at": datetime.now().isoformat(timespec="seconds"),
        },
        "global_metrics": metrics,
        "top_kept_roles": kept_titles.most_common(25),
        "domain_summary_avg_kept": dict(sorted(domain_summary.items(), key=lambda x: (-x[1], x[0]))),
        "sample_cases": samples,
    }

    base = jsonl_path.stem
    report_json = report_dir / f"{base}_report.json"
    report_txt = report_dir / f"{base}_report.txt"

    with report_json.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    with report_txt.open("w", encoding="utf-8") as f:
        f.write("=== GLOBAL METRICS ===\n")
        for k, v in metrics.items():
            f.write(f"{k}: {v}\n")

        f.write("\n=== TOP KEPT OCCUPATIONS ===\n")
        for title, cnt in kept_titles.most_common(25):
            f.write(f"{cnt}  {title}\n")

        f.write("\n=== DOMAIN AVG KEPT ===\n")
        for d, v in sorted(domain_summary.items(), key=lambda x: (-x[1], x[0])):
            f.write(f"{d}: {v}\n")

        f.write("\n=== SAMPLE CASES (truncated) ===\n")
        for s in samples[:10]:
            f.write(f"\njob_id: {s['job_id']}\n")
            f.write(f"title:  {s.get('job_ad_title')}\n")
            f.write(f"domain: {s.get('domain')} | sector: {s.get('sector')}\n")
            f.write(f"kept:   {s.get('kept')}\n")
            f.write(f"drop:   {s.get('dropped')}\n")

    print("\nSaved:")
    print(" ", report_json)
    print(" ", report_txt)

    return report_json, report_txt


full_block = f"\nFULL AD EXCERPT (use for concrete tools/duties only):\n{full_excerpt}\n" if full_excerpt else ""

    return f"""
TASK
You are a Senior Labor Market Economist auditing SOC matches. Evaluate candidates and DROP those that are NOT functional matches for the job.

GOAL: THE UP TO 4 TARGET
There are {n_candidates} candidates.
- Your goal is to keep 2 to 4 candidates for every job.
- Assume that at least 2 candidates should normally be KEPT.
- Only keep 1 if you are really sure all others are CLEARLY wrong.
- If more than 4 are valid, drop the most generic ones to fit the 4-candidate cap.

DEFAULT BEHAVIOUR
- When in doubt, KEEP rather than DROP, as long as the candidate is in the correct functional family.
- Actively look for a SECOND and THIRD valid match before dropping.

JOB CONTEXT (SOURCE OF TRUTH)
Title: {job_ad_title}
Sector/Domain: {job_sector_category} | {domain}
Functional Tasks: {tasks_str}
{full_block}

CANDIDATES (1-based)
{numbered}

ANCHOR PROTECTION (FUNCTION FIRST)
1) The ANCHOR is the candidate whose FUNCTION best matches the title’s role type (not seniority, not niche specialism).
2) You MUST KEEP the ANCHOR unless CORE EVIDENCE explicitly contradicts it.
3) TITLE KEYWORD LOCK: If the job title contains a functional keyword (e.g. driver, nurse, electrician, developer, sommelier), and a candidate directly matching that function exists, you MUST keep it.

KEEPING LOGIC (JOB MIX RECALL)
After keeping the ANCHOR, you SHOULD normally keep 2–4 additional candidates.

Keep an additional candidate if ANY apply:
- Same functional family (e.g. sommelier <-> waiter; care worker <-> home health aide).
- Same occupation at a different level (e.g. engineer <-> technologist/technician).
- Tasks partially overlap or describe adjacent duties.
- Do NOT keep roles that are purely generic or cross-sector (e.g. general managers, administrators, laborers) unless the tasks clearly align.
- The role plausibly spans more than one SOC in real labour markets.

HIERARCHY & IT GATES
1) MANAGER RULE
   - If the title does NOT include Manager/Lead/Director, only keep a managerial candidate if tasks explicitly describe staff oversight, rotas, or budgeting.
   - Do NOT keep managerial SOCs based on seniority alone.

2) IT DOMAIN LOCK
   - If the title is explicitly IT, do NOT drop the IT anchor regardless of domain.
   - For non-IT domains, keep tech roles if:
     - tasks mention specific tools (Python, SQL, APIs, etc), OR
     - the title strongly implies a digital/technical function (analyst, systems, platform, data).


RE-ANCHOR (FINAL CHECK)
Title: {job_ad_title}
Tasks: {tasks_str}
Count: {n_candidates} candidates.
You should keep 2–4 candidates.
Keeping only 1 should be rare and requires clear mismatch for all others.

OUTPUT
Return ONLY a valid JSON object with exactly one key: "drop".
Schema:
{{"drop":[...]}}
""".strip()

## bge

In [1]:
import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_bge.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
bge_jobid = jobid

JOBID: 2189695


In [ ]:
!scancel 218965

## e5

In [3]:
import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_e5.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
e5_jobid = jobid

JOBID: 2189162


## gte

In [4]:
import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_gte.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
gte_jobid = jobid

JOBID: 2189163


In [5]:
# monitoring

In [9]:
import time, subprocess
from pathlib import Path

LOG_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/logs")
def sh(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout.strip()
while True:
    print("\n=== squeue ===")
    sq = sh(f"squeue -j {jobid}")
    print(sq if sq else "<finished>")

    out_logs = sorted(LOG_DIR.glob(f"*{jobid}*.out"))
    err_logs = sorted(LOG_DIR.glob(f"*{jobid}*.err"))

    if out_logs:
        print("\n--- STDOUT ---")
        print(sh(f"tail -n 20 {out_logs[-1]}"))

    if err_logs:
        print("\n--- STDERR ---")
        print(sh(f"tail -n 20 {err_logs[-1]}"))

    if not sq:
        break

    time.sleep(5)



=== squeue ===
<finished>

--- STDOUT ---
NPZ_DIR=/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/gte_large
total 7,0M
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 569K fev  5 16:02 adzuna_month01.npz
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 570K fev  5 16:03 adzuna_month02.npz
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 571K fev  5 16:03 adzuna_month03.npz
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 571K fev  5 16:04 adzuna_month04.npz
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 569K fev  5 16:04 adzuna_month05.npz
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 571K fev  5 16:05 adzuna_month06.npz
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 565K fev  5 16:05 adzuna_month07.npz
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 562K fev  5 16:05 adzuna_month08.npz
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 562K fev  5 16:06 adzuna_month09.npz
EMBED=gte_large
[TASK] 9 -> adzuna_month10
[NPZ]  /projects/a

In [11]:
import os
import json
from pathlib import Path

# Base directory for your production runs
BASE_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection")

# Models to check
MODELS = ["bge_large", "e5_large", "gte_large"]

def print_most_recent_outputs():
    print(f"{'MODEL':<10} | {'LATEST FILENAME'}")
    print("-" * 60)
    
    for model_dir_name in MODELS:
        # Construct path to the month folder
        target_path = BASE_DIR / model_dir_name / "adzuna_month01"
        
        if not target_path.exists():
            print(f"{model_dir_name:<10} | Path not found: {target_path}")
            continue
            
        # Get all jsonl files in the directory
        files = list(target_path.glob("*.jsonl"))
        
        if not files:
            print(f"{model_dir_name:<10} | No .jsonl files found.")
            continue
            
        # Find the most recent file based on modification time (mtime)
        latest_file = max(files, key=os.path.getmtime)
        
        print(f"{model_dir_name:<10} | {latest_file.name}")
        
        # Optional: Print the last 2 records from this specific latest file
        try:
            with open(latest_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                if lines:
                    last_record = json.loads(lines[-1])
                    print(f"   -> Last Job ID: {last_record.get('job_id')}")
                    print(f"   -> Kept: {last_record.get('final')}")
        except Exception as e:
            print(f"   -> Error reading file: {e}")
        print("-" * 60)

if __name__ == "__main__":
    print_most_recent_outputs()

MODEL      | LATEST FILENAME
------------------------------------------------------------
bge_large  | llama_drop_only_adzuna_month01_0_1000_job2189164_task0_20260206_225108.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['First-Line Supervisors of Retail Sales Workers', 'Logisticians', 'First-Line Supervisors of Non-Retail Sales Workers', 'Sales Managers']
------------------------------------------------------------
e5_large   | llama_drop_only_adzuna_month01_0_1000_job2189177_task0_20260206_225057.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['Sales Managers']
------------------------------------------------------------
gte_large  | llama_drop_only_adzuna_month01_0_1000_job2189190_task0_20260206_225108.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['Stockers and Order Fillers', 'First-Line Supervisors of Non-Retail Sales Workers', 'Transportation, Storage, and Distribution Managers', 'First-Line Supervisors of Retail Sales Workers']
---------------------------------------------

## Results comparison 14 months vs 3 embeddings models selected by LLAMA

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn3
from pathlib import Path
import json
import gc # Garbage Collection interface

# ---- CONFIGURATION ----
BASE_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection")

# Stop loading after this many jobs per model to prevent crash.
# Set to None if you have 64GB+ RAM. 
MAX_JOBS = 200000 

# Define Model Base Folders
MODEL_FOLDERS = {
    "BGE": "bge_large",
    "E5":  "e5_large",
    "GTE": "gte_large"
}

# Define Month Range (1 to 14)
MONTHS = range(1, 15) # 1 through 14

def load_all_months_safe(base_dir, model_folder, months, limit=None):
    """
    Loops through adzuna_month01 to adzuna_month14, aggregating data.
    """
    aggregated_data = {}
    total_files_processed = 0
    count = 0
    
    print(f"\n--> Loading Model: {model_folder} (Months 01-14)")

    for m in months:
        # Construct path: e.g., .../bge_large/adzuna_month01
        month_dir = base_dir / model_folder / f"adzuna_month{m:02d}"
        
        if not month_dir.exists():
            print(f"    [Skip] {month_dir.name} not found.")
            continue

        files = list(month_dir.glob("*.jsonl"))
        if not files:
            continue

        print(f"    Processing {month_dir.name} ({len(files)} files)...")
        
        for filepath in files:
            if limit and count >= limit:
                print(f"    [SAFETY STOP] Limit of {limit} jobs reached.")
                return aggregated_data
                
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    for line in f:
                        if not line.strip(): continue
                        
                        obj = json.loads(line)
                        jid = obj['job_id']
                        
                        current_set = set(obj.get('final', []))
                        
                        # Store/Update data
                        if jid in aggregated_data:
                            aggregated_data[jid].update(current_set)
                        else:
                            aggregated_data[jid] = current_set
                            count += 1
                            
                        # Check limit periodically (every 1000 lines) to stay fast
                        if limit and count >= limit and count % 1000 == 0:
                            break
                            
            except Exception as e:
                print(f"    Error reading {filepath.name}: {e}")
                
            total_files_processed += 1

    print(f"    Done. Loaded {len(aggregated_data)} unique jobs from {total_files_processed} files.")
    return aggregated_data

# 1. Load Data (Looping through Months)
results = {}
for name, folder in MODEL_FOLDERS.items():
    results[name] = load_all_months_safe(BASE_DIR, folder, MONTHS, limit=MAX_JOBS)

loaded = [n for n, d in results.items() if d]
if not loaded:
    print("No data loaded. Check paths.")
    exit()

# 2. Find Common Job IDs
print("\n--- Calculating Intersections ---")
common_ids = set.intersection(*[set(results[m].keys()) for m in loaded])
print(f"Common Job IDs across all models: {len(common_ids)}")

if len(common_ids) == 0:
    print("No common jobs found. Exiting.")
    exit()

# 3. Build Report Data
print("--- Building Comparison DataFrame ---")
report_data = []

for jid in common_ids:
    bge_set = results["BGE"].get(jid, set())
    e5_set = results["E5"].get(jid, set())
    gte_set = results["GTE"].get(jid, set())
    
    all_agree = (bge_set == e5_set == gte_set)
    
    outlier = "None"
    if bge_set == e5_set != gte_set: outlier = "GTE"
    elif bge_set == gte_set != e5_set: outlier = "E5"
    elif e5_set == gte_set != bge_set: outlier = "BGE"
    elif not all_agree: outlier = "All Diverged"

    report_data.append({
        "job_id": jid,
        "all_agree": all_agree,
        "outlier": outlier,
        "bge_count": len(bge_set),
        "e5_count": len(e5_set),
        "gte_count": len(gte_set),
        "consensus_count": len(bge_set & e5_set & gte_set)
    })

df = pd.DataFrame(report_data)

# FREE MEMORY
del report_data 
gc.collect()

# 4. Stats
print("\n=== COMPREHENSIVE COMPARISON REPORT (Months 01-14) ===")
print(f"Total Common Jobs: {len(df)}")
print(f"Absolute Consensus: {df['all_agree'].sum()} ({df['all_agree'].mean():.1%})")
print("\n--- Outlier Frequency ---")
print(df[df['outlier'] != "None"]['outlier'].value_counts())

# 5. Visualizations
print("\n--- Generating Plots ---")

# A. Jaccard Heatmap
model_names = list(results.keys())
j_matrix = np.zeros((3, 3))

for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        if i == j:
            j_matrix[i, j] = 1.0
            continue
            
        total_score = 0
        count = 0
        for jid in common_ids:
            s1 = results[m1][jid]
            s2 = results[m2][jid]
            u_len = len(s1 | s2)
            total_score += (len(s1 & s2) / u_len) if u_len > 0 else 1.0
            count += 1
        j_matrix[i, j] = total_score / count if count > 0 else 0

plt.figure(figsize=(8, 6))
sns.heatmap(j_matrix, annot=True, xticklabels=model_names, yticklabels=model_names, cmap="YlGnBu", vmin=0, vmax=1)
plt.title("Avg. Jaccard Similarity (14-Month Aggregated)")
plt.tight_layout()
plt.savefig("semantic_overlap_heatmap_14months.png")
plt.close()

# FREE MEMORY
del df
gc.collect()

# B. Venn Diagram
print("Building Venn Diagram sets...")
try:
    venn_sets = []
    for name in model_names:
        s = set()
        for jid in common_ids:
            for occ in results[name][jid]:
                s.add(f"{jid}|{occ}") 
        venn_sets.append(s)

    plt.figure(figsize=(10, 8))
    venn3(venn_sets, set_labels=model_names)
    plt.title("Consensus Core vs. Variations (14 Months)")
    plt.tight_layout()
    plt.savefig("assignment_venn_14months.png")
    plt.close()
    print("Success: Plots saved.")

except MemoryError:
    print("Skipped Venn: Not enough memory.")
except Exception as e:
    print(f"Skipped Venn: {e}")

print("\nDone.")


--> Loading Model: bge_large (Months 01-14)
    Processing adzuna_month01 (1 files)...
    Processing adzuna_month02 (1 files)...
    Processing adzuna_month03 (1 files)...
    Processing adzuna_month04 (1 files)...
    Processing adzuna_month05 (1 files)...
    Processing adzuna_month06 (1 files)...
    Processing adzuna_month07 (1 files)...
    Processing adzuna_month08 (1 files)...
    Processing adzuna_month09 (1 files)...
    Processing adzuna_month10 (1 files)...
    Processing adzuna_month11 (1 files)...
    Processing adzuna_month12 (1 files)...
    Processing adzuna_month13 (1 files)...
    Processing adzuna_month14 (1 files)...
    Done. Loaded 13996 unique jobs from 14 files.

--> Loading Model: e5_large (Months 01-14)
    Processing adzuna_month01 (1 files)...
    Processing adzuna_month02 (1 files)...
    Processing adzuna_month03 (1 files)...
    Processing adzuna_month04 (1 files)...
    Processing adzuna_month05 (1 files)...
    Processing adzuna_month06 (1 files)...
 

In [20]:
import json
import re
import statistics
import random
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
BASE_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection")
OUTPUT_FILE = "FULL_TEXT_REPORT.txt"

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
IT_PATTERNS = [
    r"\bsoftware\b", r"\bdeveloper\b", r"\bengineer\b", r"\bdata\b", r"\bml\b", r"\bai\b",
    r"\bcloud\b", r"\bcyber\b", r"\bsecurity\b", r"\bnetwork\b", r"\bsystems?\b",
    r"\bdatabase\b", r"\bit\b", r"\bdevops\b"
]
_IT_RE = re.compile("|".join(IT_PATTERNS), flags=re.I)

def _is_it_role(title: str) -> bool:
    if not title: return False
    return bool(_IT_RE.search(title))

def _infer_npz(jsonl_path: Path) -> Path:
    # Try to find the month folder name (e.g., adzuna_month01)
    # Structure: .../model_name/adzuna_monthXX/file.jsonl
    month = jsonl_path.parent.name
    # NPZ is usually at .../model_name/adzuna_monthXX.npz
    return jsonl_path.parent.parent / f"{month}.npz"

def _load_npz_lookup(npz_path: Path):
    if not npz_path.exists(): return {}
    try:
        with np.load(npz_path, allow_pickle=True) as z:
            files = z.files
            job_id_key = next((k for k in ["job_ids", "job_id"] if k in files), None)
            if not job_id_key: return {}
            
            # Extract arrays
            jids = z[job_id_key].astype(str)
            
            def get_col(candidates):
                k = next((x for x in candidates if x in files), None)
                return z[k] if k else [None] * len(jids)

            titles = get_col(["job_ad_title", "title"])
            domains = get_col(["domain"])
            sectors = get_col(["job_sector_category", "sector"])

        lookup = {}
        for i, jid in enumerate(jids):
            lookup[jid] = {
                "title": str(titles[i]) if titles[i] is not None else "N/A",
                "domain": str(domains[i]) if domains[i] is not None else "UNKNOWN",
                "sector": str(sectors[i]) if sectors[i] is not None else "UNKNOWN"
            }
        return lookup
    except Exception as e:
        print(f"Error loading NPZ {npz_path}: {e}")
        return {}

# ---------------------------------------------------------------------
# AGGREGATOR CLASS
# ---------------------------------------------------------------------
class ModelStats:
    def __init__(self, name):
        self.name = name
        self.before_counts = []
        self.after_counts = []
        self.empty_outputs = 0
        self.total_kept = 0
        self.it_leak = 0
        self.kept_titles = Counter()
        self.domain_kept_counts = defaultdict(list)
        self.samples = []
    
    def add_job(self, jid, candidates, final, ctx):
        b_len = len(candidates) if candidates else len(final) # Fallback
        a_len = len(final)
        
        self.before_counts.append(b_len)
        self.after_counts.append(a_len)
        
        if a_len == 0: self.empty_outputs += 1
        
        for t in final:
            self.kept_titles[t] += 1
            self.total_kept += 1
            if _is_it_role(t): self.it_leak += 1
            
        dom = ctx.get("domain", "UNKNOWN")
        self.domain_kept_counts[dom].append(a_len)

        # Sampling logic (keep ~10 diverse examples)
        if len(self.samples) < 10 and random.random() < 0.01:
            self.samples.append({
                "job_id": jid,
                "title": ctx.get("title"),
                "domain": ctx.get("domain"),
                "sector": ctx.get("sector"),
                "kept": final,
                "drop": [] # We don't have drop in this simplistic pass, or need to calc it
                # Note: To get 'drop', we need candidate list from JSON or Context. 
                # If JSON has 'drop' key, use it.
            })

# ---------------------------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------------------------
def generate_full_report():
    # 1. Initialize Containers
    # We map folder names to clean display names
    models = {
        "bge_large": ModelStats("BGE_LARGE"),
        "e5_large":  ModelStats("E5_LARGE"),
        "gte_large": ModelStats("GTE_LARGE")
    }

    print(f"Scanning {BASE_DIR}...")
    files = list(BASE_DIR.rglob("llama_drop_only_*.jsonl"))
    print(f"Found {len(files)} files.")

    # 2. Process Files
    for fpath in files:
        # Identify model from path
        model_key = next((k for k in models if k in fpath.parts), None)
        if not model_key: continue # Skip if not one of our target models

        stats = models[model_key]
        npz_path = _infer_npz(fpath)
        lookup = _load_npz_lookup(npz_path)
        
        print(f"Processing {fpath.name} ({stats.name})...")

        with fpath.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                try:
                    obj = json.loads(line)
                    jid = str(obj.get("job_id", ""))
                    if not jid: continue
                    
                    ctx = lookup.get(jid, {})
                    cand = obj.get("candidates", [])
                    final = obj.get("final", [])
                    dropped = obj.get("drop", []) # Capture dropped for samples

                    stats.add_job(jid, cand, final, ctx)
                    
                    # Update last sample with real dropped data if just added
                    if stats.samples and stats.samples[-1]["job_id"] == jid:
                        stats.samples[-1]["drop"] = dropped
                except: continue

    # 3. Generate Report String
    final_output = []

    for key, stats in models.items():
        if not stats.before_counts: continue

        # --- Calculate Metrics ---
        avg_before = statistics.mean(stats.before_counts)
        avg_after = statistics.mean(stats.after_counts)
        drop_rate = 1.0 - (avg_after / avg_before) if avg_before > 0 else 0.0
        it_share = stats.it_leak / max(stats.total_kept, 1)
        
        # --- Header ---
        final_output.append("=" * 60)
        final_output.append(f"****** {stats.name} ******")
        final_output.append("=" * 60)
        
        # --- Global Metrics ---
        final_output.append("=== GLOBAL METRICS ===")
        final_output.append(f"jobs: {len(stats.before_counts)}")
        final_output.append(f"avg_candidates_before: {avg_before:.3f}")
        final_output.append(f"avg_candidates_after: {avg_after:.3f}")
        final_output.append(f"drop_rate: {drop_rate:.4f}")
        final_output.append(f"empty_outputs_percent: {(100 * stats.empty_outputs / len(stats.before_counts)):.1f}")
        final_output.append(f"it_leakage_share: {it_share:.4f}")
        final_output.append(f"min_kept: {min(stats.after_counts)}")
        final_output.append(f"max_kept: {max(stats.after_counts)}")
        final_output.append("")

        # --- Top Kept ---
        final_output.append("=== TOP KEPT OCCUPATIONS ===")
        for title, count in stats.kept_titles.most_common(25):
            final_output.append(f"{count}  {title}")
        final_output.append("")

        # --- Domain Avg ---
        final_output.append("=== DOMAIN AVG KEPT ===")
        # Sort by Avg (Desc), then Name (Asc)
        domain_avgs = {k: statistics.mean(v) for k, v in stats.domain_kept_counts.items()}
        sorted_domains = sorted(domain_avgs.items(), key=lambda x: (-x[1], x[0]))
        
        for dom, val in sorted_domains:
            # Format: 3 or 2.5 or 2.857 (remove trailing zeros if integer)
            val_str = f"{val:.3f}".rstrip('0').rstrip('.')
            final_output.append(f"{dom}: {val_str}")
        final_output.append("")

        # --- Samples ---
        final_output.append("=== SAMPLE CASES (truncated) ===")
        for s in stats.samples:
            final_output.append(f"\njob_id: {s['job_id']}")
            final_output.append(f"title:  {s['title']}")
            final_output.append(f"domain: {s['domain']} | sector: {s['sector']}")
            final_output.append(f"kept:   {s['kept']}")
            final_output.append(f"drop:   {s['drop']}")
        final_output.append("\n\n")

    # 4. Save
    full_text = "\n".join(final_output)
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(full_text)
    
    print("\n" + "="*40)
    print(f"Report Generated: {OUTPUT_FILE}")
    print("="*40)
    # Print preview
    print(full_text[:100000])

if __name__ == "__main__":
    generate_full_report()

Scanning /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection...
Found 42 files.
Processing llama_drop_only_adzuna_month13_0_1000_job2189189_task12_20260206_225108.jsonl (E5_LARGE)...
Processing llama_drop_only_adzuna_month05_0_1000_job2189181_task4_20260206_225110.jsonl (E5_LARGE)...
Processing llama_drop_only_adzuna_month03_0_1000_job2189179_task2_20260206_225107.jsonl (E5_LARGE)...
Processing llama_drop_only_adzuna_month11_0_1000_job2189187_task10_20260206_225111.jsonl (E5_LARGE)...
Processing llama_drop_only_adzuna_month04_0_1000_job2189180_task3_20260206_225107.jsonl (E5_LARGE)...
Processing llama_drop_only_adzuna_month10_0_1000_job2189186_task9_20260206_225111.jsonl (E5_LARGE)...
Processing llama_drop_only_adzuna_month02_0_1000_job2189178_task1_20260206_225107.jsonl (E5_LARGE)...
Processing llama_drop_only_adzuna_month07_0_1000_job2189183_task6_20260206_225110.jsonl (E5_LARGE)...
Processing llama_drop_only_adzuna_month06_0

In [13]:
# 1. Define once (Update these only)

notebook_name ='01_prompt_14_months'

import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
import gc  # Garbage collection

# ---- 1. CONFIGURATION ----
BASE_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection")

MODELS = {
    "BGE": "bge_large",
    "E5":  "e5_large",
    "GTE": "gte_large"
}

# Range of months to process
MONTHS = range(1, 15)  # 1 to 14

# Sampling settings
SAMPLES_PER_MONTH = 10  # 10 jobs * 14 months = ~140 jobs per model

# ---- 2. HELPER FUNCTIONS ----
def _to_str_array(x: np.ndarray) -> np.ndarray:
    if x.dtype == object:
        return np.array([v.decode("utf-8") if isinstance(v, (bytes, bytearray)) else str(v) for v in x])
    if np.issubdtype(x.dtype, np.bytes_):
        return np.array([v.decode("utf-8") for v in x])
    return x.astype(str)

def load_meta_from_npz(npz_path: Path) -> pd.DataFrame:
    if not npz_path.exists():
        return pd.DataFrame()
        
    try:
        with np.load(npz_path, allow_pickle=True) as npz:
            # Flexible key checking
            files = npz.files
            job_id_key = next((k for k in ["job_id", "job_ids"] if k in files), None)
            title_key = next((k for k in ["job_ad_title", "title"] if k in files), None)
            domain_key = "domain" if "domain" in files else None
            sector_key = "job_sector_category" if "job_sector_category" in files else None

            if not job_id_key: return pd.DataFrame()

            meta = {
                "job_id": _to_str_array(npz[job_id_key]),
                "title": _to_str_array(npz[title_key]) if title_key else "",
            }
            if domain_key: meta["domain"] = _to_str_array(npz[domain_key])
            if sector_key: meta["sector"] = _to_str_array(npz[sector_key])
            
        return pd.DataFrame(meta)
    except Exception as e:
        print(f"Error loading NPZ {npz_path.name}: {e}")
        return pd.DataFrame()

def load_results_from_folder(jsonl_folder: Path) -> pd.DataFrame:
    rows = []
    # Find all .jsonl files in the month folder
    files = list(jsonl_folder.glob("*.jsonl"))
    
    if not files:
        return pd.DataFrame()

    for file_path in files:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try: 
                        obj = json.loads(line)
                        rows.append(obj)
                    except: continue
        except Exception as e:
            print(f"Error reading JSONL {file_path.name}: {e}")

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df["job_id"] = df["job_id"].astype(str)

    # Standardize column names
    if "final" in df.columns: df = df.rename(columns={"final": "kept"})
    if "drop" in df.columns: df = df.rename(columns={"drop": "dropped"})
    
    # Ensure columns exist
    if "kept" not in df.columns: df["kept"] = None
    if "dropped" not in df.columns: df["dropped"] = None

    # Helper to convert lists to string for CSV
    def _list_to_csv(x):
        if isinstance(x, list): return ", ".join(map(str, x))
        return "" if x is None else str(x)

    df["kept"] = df["kept"].apply(_list_to_csv)
    df["dropped"] = df["dropped"].apply(_list_to_csv)
    
    return df[["job_id", "kept", "dropped"]]

# ---- 3. CORE BUILDER (LOOPED) ----
def build_audit_dataset_looped(base_dir, models, months, samples_per_month=10, seed=42) -> pd.DataFrame:
    all_frames = []
    
    print(f"Starting Audit Loop (Months {months.start}-{months.stop-1})...")

    for model_name, model_folder in models.items():
        print(f"\n--- Processing Model: {model_name} ---")
        
        for m in months:
            month_str = f"adzuna_month{m:02d}"
            
            # Construct Paths
            # NPZ is usually at: model_folder/adzuna_monthXX.npz
            npz_path = base_dir / model_folder / f"{month_str}.npz"
            
            # JSONL is inside: model_folder/adzuna_monthXX/*.jsonl
            jsonl_dir = base_dir / model_folder / month_str
            
            if not jsonl_dir.exists():
                # print(f"Skipping {month_str} (Folder not found)")
                continue

            # 1. Load Data
            res_df = load_results_from_folder(jsonl_dir)
            if res_df.empty: continue
            
            meta_df = load_meta_from_npz(npz_path)
            # Note: If NPZ is missing, we still might want the JSONL data, 
            # but usually we need metadata for the audit. 
            # If NPZ missing, meta_df is empty.
            
            # 2. Merge
            if not meta_df.empty:
                merged = res_df.merge(meta_df, on="job_id", how="inner")
            else:
                # If no NPZ, just use JSONL data and fill missing cols
                merged = res_df
                merged["title"] = "N/A (NPZ Missing)"
                merged["domain"] = ""
                merged["sector"] = ""

            if merged.empty: continue

            # 3. Sample immediately to save memory
            sample_size = min(samples_per_month, len(merged))
            sampled = merged.sample(n=sample_size, random_state=seed).copy()
            
            sampled["model"] = model_name
            sampled["month"] = month_str # Track which month it came from
            
            all_frames.append(sampled)
            
            # Clean up
            del res_df, meta_df, merged
            gc.collect()
            
            print(f"   + {month_str}: Added {len(sampled)} jobs")

    # 4. Concatenate Final Result
    if not all_frames:
        return pd.DataFrame()

    out = pd.concat(all_frames, ignore_index=True)
    
    # Ensure all columns exist
    cols = ["job_id", "model", "month", "title", "domain", "sector", "kept", "dropped"]
    for c in cols:
        if c not in out.columns: out[c] = ""
            
    return out[cols].sort_values(["model", "month", "job_id"]).reset_index(drop=True)

# ---- 4. EXECUTION ----
if __name__ == "__main__":
    # 1. Run Builder
    audit_df = build_audit_dataset_looped(
        BASE_DIR, 
        MODELS, 
        MONTHS, 
        samples_per_month=SAMPLES_PER_MONTH, 
        seed=42
    )
    
    if audit_df.empty:
        print("\n[ERROR] No data found for any model/month.")
    else:
        # 2. Determine Filename
        try:
            prefix = Path(__file__).stem
        except NameError:
            prefix = "notebook_audit" # Default if running in cell
            
        out_filename = f"{prefix}_sanity_check_months_01_14.csv"
        out_path = Path(out_filename)
        
        # 3. Save
        audit_df.to_csv(out_path, index=False)
        
        print("-" * 30)
        print(f"[SUCCESS] File written: {out_path.absolute()}")
        print(f"Total Rows: {len(audit_df)}")
        print("Breakdown by Model:")
        print(audit_df['model'].value_counts())
        print("-" * 30)

Starting Audit Loop (Months 1-14)...

--- Processing Model: BGE ---
   + adzuna_month01: Added 10 jobs
   + adzuna_month02: Added 10 jobs
   + adzuna_month03: Added 10 jobs
   + adzuna_month04: Added 10 jobs
   + adzuna_month05: Added 10 jobs
   + adzuna_month06: Added 10 jobs
   + adzuna_month07: Added 10 jobs
   + adzuna_month08: Added 10 jobs
   + adzuna_month09: Added 10 jobs
   + adzuna_month10: Added 10 jobs
   + adzuna_month11: Added 10 jobs
   + adzuna_month12: Added 10 jobs
   + adzuna_month13: Added 10 jobs
   + adzuna_month14: Added 10 jobs

--- Processing Model: E5 ---
   + adzuna_month01: Added 10 jobs
   + adzuna_month02: Added 10 jobs
   + adzuna_month03: Added 10 jobs
   + adzuna_month04: Added 10 jobs
   + adzuna_month05: Added 10 jobs
   + adzuna_month06: Added 10 jobs
   + adzuna_month07: Added 10 jobs
   + adzuna_month08: Added 10 jobs
   + adzuna_month09: Added 10 jobs
   + adzuna_month10: Added 10 jobs
   + adzuna_month11: Added 10 jobs
   + adzuna_month12: Added 

In [21]:
import json
import re
import statistics
import random
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
BASE_DIR = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/"
    "AISI_demo/stage_3/dev/llm_negative_selection"
)
OUTPUT_FILE = "FULL_TEXT_REPORT.txt"

# What "good" looks like for negative selection (tweak as needed)
SCORE_CFG = {
    # Hard constraints / primary risk controls
    "empty_target": 0.0,           # percent
    "empty_hard_fail": 0.5,        # percent
    "it_leak_target": 0.05,        # share
    "it_leak_hard_fail": 0.10,     # share

    # Behavioural targets
    "avg_after_target": 2.2,       # your stated preference: return 1 only when very sure, else keep 2, up to 3
    "avg_after_tolerance": 0.6,    # acceptable band around target
    "max_kept_soft": 3,            # prefer <=3
    "max_kept_hard": 4,            # observed; if >4 something is off

    # Stability / distribution checks (soft but useful)
    "top1_conc_target": 0.06,      # share of all kept titles that are the single most common
    "domain_disp_target": 0.55,    # std dev across domain mean-kept (lower is better)
}

# Weights for composite score (must sum to 1.0 ideally, but not required)
SCORE_WEIGHTS = {
    "empty": 0.30,
    "it_leak": 0.30,
    "avg_after": 0.15,
    "max_kept": 0.10,
    "top1_conc": 0.07,
    "domain_disp": 0.08,
}

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
IT_PATTERNS = [
    r"\bsoftware\b", r"\bdeveloper\b", r"\bengineer\b", r"\bdata\b", r"\bml\b", r"\bai\b",
    r"\bcloud\b", r"\bcyber\b", r"\bsecurity\b", r"\bnetwork\b", r"\bsystems?\b",
    r"\bdatabase\b", r"\bit\b", r"\bdevops\b"
]
_IT_RE = re.compile("|".join(IT_PATTERNS), flags=re.I)

def _is_it_role(title: str) -> bool:
    return bool(title and _IT_RE.search(title))

def _infer_npz(jsonl_path: Path) -> Path:
    month = jsonl_path.parent.name
    return jsonl_path.parent.parent / f"{month}.npz"

def _load_npz_lookup(npz_path: Path):
    if not npz_path.exists():
        return {}
    try:
        with np.load(npz_path, allow_pickle=True) as z:
            files = set(z.files)
            job_id_key = next((k for k in ["job_ids", "job_id"] if k in files), None)
            if not job_id_key:
                return {}

            jids = z[job_id_key].astype(str)

            def get_col(candidates):
                k = next((x for x in candidates if x in files), None)
                if not k:
                    return [None] * len(jids)
                return z[k]

            titles = get_col(["job_ad_title", "title"])
            domains = get_col(["domain"])
            sectors = get_col(["job_sector_category", "sector"])

        lookup = {}
        for i, jid in enumerate(jids):
            lookup[jid] = {
                "title": str(titles[i]) if titles[i] is not None else "N/A",
                "domain": str(domains[i]) if domains[i] is not None else "UNKNOWN",
                "sector": str(sectors[i]) if sectors[i] is not None else "UNKNOWN",
            }
        return lookup
    except Exception as e:
        print(f"Error loading NPZ {npz_path}: {e}")
        return {}

def _clip01(x: float) -> float:
    if x < 0.0:
        return 0.0
    if x > 1.0:
        return 1.0
    return x

def _penalty_over_target(value: float, target: float, scale: float) -> float:
    """
    Returns a [0,1] penalty.
    0 if value <= target.
    1 if value >= target + scale.
    """
    if value <= target:
        return 0.0
    return _clip01((value - target) / max(scale, 1e-9))

def _penalty_abs_deviation(value: float, target: float, tol: float, scale: float) -> float:
    """
    0 if within tolerance band around target.
    Linearly increases to 1 outside tolerance band by 'scale'.
    """
    dev = abs(value - target)
    if dev <= tol:
        return 0.0
    return _clip01((dev - tol) / max(scale, 1e-9))

# ---------------------------------------------------------------------
# AGGREGATOR CLASS
# ---------------------------------------------------------------------
class ModelStats:
    def __init__(self, name):
        self.name = name
        self.before_counts = []
        self.after_counts = []
        self.empty_outputs = 0
        self.total_kept = 0
        self.it_leak = 0
        self.kept_titles = Counter()
        self.domain_kept_counts = defaultdict(list)
        self.samples = []

    def add_job(self, jid, candidates, final, dropped, ctx):
        b_len = len(candidates) if candidates is not None else 0
        a_len = len(final) if final is not None else 0

        self.before_counts.append(b_len)
        self.after_counts.append(a_len)

        if a_len == 0:
            self.empty_outputs += 1

        for t in final or []:
            self.kept_titles[t] += 1
            self.total_kept += 1
            if _is_it_role(t):
                self.it_leak += 1

        dom = ctx.get("domain", "UNKNOWN")
        self.domain_kept_counts[dom].append(a_len)

        # Reservoir-ish sampling with mild diversity: prefer new domains early
        if len(self.samples) < 12:
            if random.random() < 0.02:
                self.samples.append({
                    "job_id": jid,
                    "title": ctx.get("title"),
                    "domain": ctx.get("domain"),
                    "sector": ctx.get("sector"),
                    "kept": final,
                    "drop": dropped,
                })

# ---------------------------------------------------------------------
# SCORING
# ---------------------------------------------------------------------
def score_model(stats: ModelStats):
    jobs = len(stats.after_counts)
    if jobs == 0:
        return None

    avg_before = statistics.mean(stats.before_counts) if stats.before_counts else 0.0
    avg_after = statistics.mean(stats.after_counts) if stats.after_counts else 0.0
    drop_rate = 1.0 - (avg_after / avg_before) if avg_before > 0 else 0.0
    empty_pct = 100.0 * stats.empty_outputs / max(jobs, 1)
    it_share = stats.it_leak / max(stats.total_kept, 1)

    max_kept = max(stats.after_counts) if stats.after_counts else 0
    min_kept = min(stats.after_counts) if stats.after_counts else 0

    # Concentration: how dominant is the most frequent kept occupation
    top1 = stats.kept_titles.most_common(1)
    top1_share = (top1[0][1] / stats.total_kept) if (top1 and stats.total_kept) else 0.0

    # Domain dispersion: std dev of domain mean kept counts (lower = more consistent)
    domain_means = []
    for dom, vals in stats.domain_kept_counts.items():
        if vals:
            domain_means.append(statistics.mean(vals))
    domain_disp = statistics.pstdev(domain_means) if len(domain_means) >= 2 else 0.0

    # Penalties in [0,1]
    p_empty = _penalty_over_target(empty_pct, SCORE_CFG["empty_target"], SCORE_CFG["empty_hard_fail"])
    p_it = _penalty_over_target(it_share, SCORE_CFG["it_leak_target"], SCORE_CFG["it_leak_hard_fail"] - SCORE_CFG["it_leak_target"])
    p_avg_after = _penalty_abs_deviation(
        avg_after,
        SCORE_CFG["avg_after_target"],
        SCORE_CFG["avg_after_tolerance"],
        scale=1.0,  # 1 candidate beyond tolerance = full penalty
    )

    # max_kept penalty: 0 if <= soft, 1 if >= hard
    if max_kept <= SCORE_CFG["max_kept_soft"]:
        p_max_kept = 0.0
    else:
        p_max_kept = _clip01((max_kept - SCORE_CFG["max_kept_soft"]) / max(SCORE_CFG["max_kept_hard"] - SCORE_CFG["max_kept_soft"], 1e-9))

    p_top1 = _penalty_over_target(top1_share, SCORE_CFG["top1_conc_target"], scale=0.10)  # 10pp beyond target = full penalty
    p_domain = _penalty_over_target(domain_disp, SCORE_CFG["domain_disp_target"], scale=0.80)

    # Weighted penalty and score in [0,100]
    weighted_penalty = (
        SCORE_WEIGHTS["empty"] * p_empty +
        SCORE_WEIGHTS["it_leak"] * p_it +
        SCORE_WEIGHTS["avg_after"] * p_avg_after +
        SCORE_WEIGHTS["max_kept"] * p_max_kept +
        SCORE_WEIGHTS["top1_conc"] * p_top1 +
        SCORE_WEIGHTS["domain_disp"] * p_domain
    )
    score = 100.0 * (1.0 - _clip01(weighted_penalty))

    # Flag hard fails (useful for gating)
    hard_fail = False
    reasons = []
    if empty_pct > SCORE_CFG["empty_hard_fail"]:
        hard_fail = True
        reasons.append(f"empty_pct={empty_pct:.2f}% > {SCORE_CFG['empty_hard_fail']:.2f}%")
    if it_share > SCORE_CFG["it_leak_hard_fail"]:
        hard_fail = True
        reasons.append(f"it_leak={it_share:.3f} > {SCORE_CFG['it_leak_hard_fail']:.3f}")
    if max_kept > SCORE_CFG["max_kept_hard"]:
        hard_fail = True
        reasons.append(f"max_kept={max_kept} > {SCORE_CFG['max_kept_hard']}")

    breakdown = {
        "score": score,
        "hard_fail": hard_fail,
        "hard_fail_reasons": reasons,
        "jobs": jobs,
        "avg_before": avg_before,
        "avg_after": avg_after,
        "drop_rate": drop_rate,
        "empty_pct": empty_pct,
        "it_share": it_share,
        "min_kept": min_kept,
        "max_kept": max_kept,
        "top1_share": top1_share,
        "domain_disp": domain_disp,
        "penalties": {
            "empty": p_empty,
            "it_leak": p_it,
            "avg_after": p_avg_after,
            "max_kept": p_max_kept,
            "top1_conc": p_top1,
            "domain_disp": p_domain,
        }
    }
    return breakdown

# ---------------------------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------------------------
def generate_full_report():
    random.seed(42)

    models = {
        "bge_large": ModelStats("BGE_LARGE"),
        "e5_large":  ModelStats("E5_LARGE"),
        "gte_large": ModelStats("GTE_LARGE"),
    }

    print(f"Scanning {BASE_DIR}...")
    files = list(BASE_DIR.rglob("llama_drop_only_*.jsonl"))
    print(f"Found {len(files)} files.")

    # Process files
    for fpath in files:
        model_key = next((k for k in models if k in fpath.parts), None)
        if not model_key:
            continue

        stats = models[model_key]
        npz_path = _infer_npz(fpath)
        lookup = _load_npz_lookup(npz_path)

        print(f"Processing {fpath} ({stats.name})")

        with fpath.open("r", encoding="utf-8") as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                try:
                    obj = json.loads(s)
                except Exception:
                    continue

                jid = str(obj.get("job_id", "")).strip()
                if not jid:
                    continue

                ctx = lookup.get(jid, {})
                candidates = obj.get("candidates", [])
                final = obj.get("final", [])
                dropped = obj.get("drop", [])

                # Fallback if schema differs
                if final is None:
                    final = []
                if candidates is None:
                    candidates = []
                if dropped is None:
                    dropped = []

                stats.add_job(jid, candidates, final, dropped, ctx)

    # Build report text
    lines = []
    lines.append("=" * 40)
    lines.append(f"Report Generated: {OUTPUT_FILE}")
    lines.append("=" * 40)
    lines.append(f"Generated at: {datetime.now().isoformat(timespec='seconds')}")
    lines.append("")

    # Scorecard first, so you do not have to eyeball it
    lines.append("=" * 60)
    lines.append("=== MODEL SCORECARD (composite) ===")
    lines.append("=" * 60)

    score_rows = []
    for key, stats in models.items():
        br = score_model(stats)
        if br is None:
            continue
        score_rows.append((stats.name, br))

    score_rows.sort(key=lambda x: (x[1]["hard_fail"], -x[1]["score"]))

    if not score_rows:
        lines.append("No model results found. Check BASE_DIR and filename patterns.")
        full_text = "\n".join(lines)
        Path(OUTPUT_FILE).write_text(full_text, encoding="utf-8")
        print(full_text)
        return

    for name, br in score_rows:
        hf = "YES" if br["hard_fail"] else "NO"
        lines.append(
            f"{name}: score={br['score']:.1f} | hard_fail={hf} | "
            f"it={br['it_share']:.4f} | empty={br['empty_pct']:.2f}% | "
            f"avg_after={br['avg_after']:.3f} | max_kept={br['max_kept']} | "
            f"top1_share={br['top1_share']:.3f} | domain_disp={br['domain_disp']:.3f}"
        )
        if br["hard_fail_reasons"]:
            lines.append(f"  hard_fail_reasons: {', '.join(br['hard_fail_reasons'])}")
        p = br["penalties"]
        lines.append(
            "  penalties: "
            f"empty={p['empty']:.2f}, it={p['it_leak']:.2f}, avg_after={p['avg_after']:.2f}, "
            f"max_kept={p['max_kept']:.2f}, top1={p['top1_conc']:.2f}, domain={p['domain_disp']:.2f}"
        )

    best = next((x for x in score_rows if not x[1]["hard_fail"]), score_rows[0])
    lines.append("")
    lines.append(f"BEST_MODEL_BY_SCORE: {best[0]} (score={best[1]['score']:.1f})")
    lines.append("")

    # Per-model detailed sections (same as before, plus the score breakdown)
    for key, stats in models.items():
        if not stats.before_counts:
            continue

        br = score_model(stats)

        avg_before = statistics.mean(stats.before_counts)
        avg_after = statistics.mean(stats.after_counts)
        drop_rate = 1.0 - (avg_after / avg_before) if avg_before > 0 else 0.0
        it_share = stats.it_leak / max(stats.total_kept, 1)

        lines.append("=" * 60)
        lines.append(f"****** {stats.name} ******")
        lines.append("=" * 60)

        if br:
            lines.append("=== SCORE BREAKDOWN ===")
            lines.append(f"composite_score: {br['score']:.2f}")
            lines.append(f"hard_fail: {br['hard_fail']}")
            if br["hard_fail_reasons"]:
                lines.append(f"hard_fail_reasons: {', '.join(br['hard_fail_reasons'])}")
            lines.append(
                "penalties: "
                f"empty={br['penalties']['empty']:.3f}, "
                f"it_leak={br['penalties']['it_leak']:.3f}, "
                f"avg_after={br['penalties']['avg_after']:.3f}, "
                f"max_kept={br['penalties']['max_kept']:.3f}, "
                f"top1_conc={br['penalties']['top1_conc']:.3f}, "
                f"domain_disp={br['penalties']['domain_disp']:.3f}"
            )
            lines.append("")

        lines.append("=== GLOBAL METRICS ===")
        lines.append(f"jobs: {len(stats.before_counts)}")
        lines.append(f"avg_candidates_before: {avg_before:.3f}")
        lines.append(f"avg_candidates_after: {avg_after:.3f}")
        lines.append(f"drop_rate: {drop_rate:.4f}")
        lines.append(f"empty_outputs_percent: {(100 * stats.empty_outputs / len(stats.before_counts)):.2f}")
        lines.append(f"it_leakage_share: {it_share:.4f}")
        lines.append(f"min_kept: {min(stats.after_counts)}")
        lines.append(f"max_kept: {max(stats.after_counts)}")
        lines.append("")

        lines.append("=== TOP KEPT OCCUPATIONS ===")
        for title, count in stats.kept_titles.most_common(25):
            lines.append(f"{count}  {title}")
        lines.append("")

        lines.append("=== DOMAIN AVG KEPT ===")
        domain_avgs = {k: statistics.mean(v) for k, v in stats.domain_kept_counts.items()}
        sorted_domains = sorted(domain_avgs.items(), key=lambda x: (-x[1], x[0]))
        for dom, val in sorted_domains:
            val_str = f"{val:.3f}".rstrip("0").rstrip(".")
            lines.append(f"{dom}: {val_str}")
        lines.append("")

        lines.append("=== SAMPLE CASES (truncated) ===")
        for s in stats.samples[:10]:
            lines.append("")
            lines.append(f"job_id: {s['job_id']}")
            lines.append(f"title:  {s['title']}")
            lines.append(f"domain: {s['domain']} | sector: {s['sector']}")
            lines.append(f"kept:   {s['kept']}")
            lines.append(f"drop:   {s['drop']}")
        lines.append("\n")

    full_text = "\n".join(lines)
    Path(OUTPUT_FILE).write_text(full_text, encoding="utf-8")

    print("=" * 40)
    print(f"Report Generated: {OUTPUT_FILE}")
    print("=" * 40)
    print(full_text[:120000])

if __name__ == "__main__":
    generate_full_report()


Scanning /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection...
Found 42 files.
Processing /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/e5_large/adzuna_month13/llama_drop_only_adzuna_month13_0_1000_job2189189_task12_20260206_225108.jsonl (E5_LARGE)
Processing /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/e5_large/adzuna_month05/llama_drop_only_adzuna_month05_0_1000_job2189181_task4_20260206_225110.jsonl (E5_LARGE)
Processing /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/e5_large/adzuna_month03/llama_drop_only_adzuna_month03_0_1000_job2189179_task2_20260206_225107.jsonl (E5_LARGE)
Processing /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/e5_large/adzuna_month11/llama_drop_o

In [22]:
import pandas as pd
import re
from collections import Counter

# ---- CONFIGURATION ----
INPUT_CSV = "notebook_audit_sanity_check_months_01_14.csv"  # Your file path

# ---- 1. CLEANING FUNCTIONS (Fixes CSV Comma Issues) ----
def clean_occupation_list(text):
    """
    Reconstructs occupation lists that were split incorrectly by CSV commas.
    e.g., "Teaching Assistants, Special Education" -> "Teaching Assistants, Special Education"
    instead of ["Teaching Assistants", "Special Education"]
    """
    if not isinstance(text, str): return []
    raw_tokens = [t.strip() for t in text.split(',')]
    cleaned = []
    
    # Known fragments that shouldn't be standalone occupations
    fragments = {
        "Hand", "General", "All Other", "Special Education", "Middle School", "Secondary School", 
        "Preschool", "Elementary", "Postsecondary", "Wholesale and Manufacturing", 
        "Technical and Scientific Products", "Except Technical and Scientific Products",
        "Except Special Education", "Except Legal", "Medical", "and Executive",
        "Except Maids and Housekeeping Cleaners", "Institution and Cafeteria", 
        "Restaurant", "Lounge", "and Coffee Shop", "Except Engines", 
        "Scientific and Technical Products", "Except Payroll and Timekeeping",
        "Family", "and School Social Workers", "Receiving", "and Inventory Clerks",
        "Stock", "and Material Movers", "Substance Abuse Social Workers",
        "Except Eligibility and Loan", "Except Epidemiologists", "Except Computer",
        "Except Home Health and Personal Care Aides", "Except Phlebotomists",
        "Except Advertising", "Except Drafters", "Except Wind and Solar",
        "Except Hydrologists and Geographers"
    }
    
    iterator = iter(raw_tokens)
    try:
        current = next(iterator)
    except StopIteration:
        return []

    cleaned.append(current)
    for token in iterator:
        is_fragment = False
        if token in fragments or token.startswith("Except ") or token.startswith("and "):
            is_fragment = True
        
        # Context-aware merging
        if token == "Family" and cleaned[-1] == "Child": is_fragment = True
        if token == "Receiving" and cleaned[-1] == "Shipping": is_fragment = True
        if token == "Stock" and cleaned[-1].endswith("Freight"): is_fragment = True
        
        if is_fragment:
            cleaned[-1] = cleaned[-1] + ", " + token
        else:
            cleaned.append(token)
            
    return set(cleaned)

# ---- 2. METRIC FUNCTIONS ----

# Stop words to ignore in word overlap (generic terms)
STOP_WORDS = {"a", "an", "the", "and", "or", "of", "for", "in", "to", 
              "manager", "senior", "junior", "assistant", "specialist", 
              "officer", "supervisor", "lead", "head", "associate"}

def get_tokens(text):
    """Extracts meaningful words from a string."""
    text = re.sub(r'[^\w\s]', '', str(text).lower())
    return {w for w in text.split() if w not in STOP_WORDS and len(w) > 2}

def calc_word_overlap(title, kept_list):
    """
    Calculates how many meaningful words from the Job Title appear in the Kept Occupations.
    Higher score = The model kept occupations that sound like the job title.
    """
    title_tokens = get_tokens(title)
    if not title_tokens: return 0.0
    
    scores = []
    for occ in kept_list:
        occ_tokens = get_tokens(occ)
        if not occ_tokens: 
            scores.append(0)
            continue
        # Intersection count
        intersection = len(title_tokens & occ_tokens)
        scores.append(intersection)
    
    if not scores: return 0
    return sum(scores) / len(scores)

# IT Keywords for Safety Check
IT_KEYWORDS = ["software", "developer", "data", "cyber", "cloud", "programmer", 
               "network", "support", "technology", "devops", "system", "full stack"]

def is_it_job(text):
    return any(k in str(text).lower() for k in IT_KEYWORDS)

def check_it_leakage(row):
    """Returns 1 if a Non-IT Job kept an IT Occupation (Bad Hallucination)."""
    title = row['title']
    
    # If the job itself is IT, we don't care (leakage is impossible)
    if is_it_job(title) or is_it_job(row.get('domain', '')):
        return 0
    
    # If job is Non-IT, check if any kept role IS IT
    for occ in row['kept_clean']:
        if is_it_job(occ):
            return 1 # Leakage detected!
            
    return 0

# ---- 3. MAIN EXECUTION ----
def generate_ranking_report():
    print(f"Loading {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    
    print("Cleaning occupation lists...")
    df['kept_clean'] = df['kept'].apply(clean_occupation_list)
    
    print("Calculating metrics...")
    # Calculate Word Overlap
    df['relevance_score'] = df.apply(lambda x: calc_word_overlap(x['title'], x['kept_clean']), axis=1)
    
    # Calculate IT Leakage
    df['is_leakage'] = df.apply(check_it_leakage, axis=1)
    
    # Filter for Non-IT jobs for the leakage denominator
    df['is_non_it_job'] = df.apply(lambda x: not (is_it_job(x['title']) or is_it_job(x.get('domain', ''))), axis=1)
    
    # Group by Model
    stats = df.groupby('model').agg(
        Total_Jobs=('job_id', 'count'),
        Avg_Word_Overlap=('relevance_score', 'mean'),
        Leakage_Count=('is_leakage', 'sum'),
        Non_IT_Jobs_Evaluated=('is_non_it_job', 'sum')
    ).reset_index()
    
    # Calculate Rate
    stats['IT_Leakage_Rate'] = stats['Leakage_Count'] / stats['Non_IT_Jobs_Evaluated']
    
    # Rank by Overlap (High is good) and Leakage (Low is good)
    stats = stats.sort_values(by=['Avg_Word_Overlap'], ascending=False)
    
    print("\n" + "="*60)
    print("FINAL MODEL RANKING REPORT")
    print("="*60)
    
    print(stats[['model', 'Avg_Word_Overlap', 'IT_Leakage_Rate', 'Leakage_Count']].to_string(index=False))
    
    print("\n--- Interpretation ---")
    print("1. Avg_Word_Overlap (Higher is Better):")
    print("   Means the model kept roles that actually match the words in the Job Title.")
    print("   (e.g., Title: 'Electrician' -> Kept: 'Electrician')")
    print("\n2. IT_Leakage_Rate (Lower is Better):")
    print("   Percentage of Non-IT jobs (e.g., Chef, Driver) where the model")
    print("   hallucinated an IT role (e.g., Software Developer).")

if __name__ == "__main__":
    generate_ranking_report()

Loading notebook_audit_sanity_check_months_01_14.csv...
Cleaning occupation lists...
Calculating metrics...

FINAL MODEL RANKING REPORT
model  Avg_Word_Overlap  IT_Leakage_Rate  Leakage_Count
   E5          0.151429         0.075000              9
  BGE          0.145000         0.100000             12
  GTE          0.129881         0.108333             13

--- Interpretation ---
1. Avg_Word_Overlap (Higher is Better):
   Means the model kept roles that actually match the words in the Job Title.
   (e.g., Title: 'Electrician' -> Kept: 'Electrician')

2. IT_Leakage_Rate (Lower is Better):
   Percentage of Non-IT jobs (e.g., Chef, Driver) where the model
   hallucinated an IT role (e.g., Software Developer).
